In [1]:
# Chi-square Tests: Categorical Features vs Signup
#
# Purpose:
# Evaluate whether high-level categorical features (browser, OS, region,
# test behavior, etc.) are statistically associated with user signup behavior.
#
# This analysis uses contingency tables and Chi-square tests of independence.
#
# Data source:
# data/visitor_features_engineered.parquet

In [4]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency

import os
os.getcwd()

'/Users/davidreynolds/projects/online_teletherapy_signups/analysis'

In [5]:
DATA_PATH = "../data/visitor_features_engineered.parquet"

df = pd.read_parquet(DATA_PATH)

df.shape

(535391, 15)

In [6]:
assert set(df["has_signup"].dropna().unique()) <= {True, False}

In [7]:
CATEGORICAL_FEATURES = [
    "browser",
    "os_name_clean",
    "sub_region",
    "test_engagement_state",
    "tests_taken_count_bucket",
    "tests_completed_count_bucket",
    "first_test_domain",
    "first_completed_test_domain",
]

In [8]:
def chi_square_test(df, feature, target="has_signup"):
    """
    Runs a Chi-square test of independence between a categorical feature
    and a binary target variable.

    Returns summary statistics and the contingency table.
    """
    contingency_table = pd.crosstab(df[feature], df[target])

    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    return {
        "feature": feature,
        "chi2": chi2,
        "dof": dof,
        "p_value": p_value,
        "significant_0_05": p_value < 0.05,
        "contingency_table": contingency_table,
        "expected_freq": expected,
    }

In [9]:
results = []

for feature in CATEGORICAL_FEATURES:
    result = chi_square_test(df, feature)
    results.append(result)

In [10]:
summary_df = (
    pd.DataFrame(results)
    .drop(columns=["contingency_table", "expected_freq"])
    .sort_values("p_value")
)

summary_df

,feature,chi2,dof,p_value,significant_0_05
3,test_engagement_state,2083.483341,2,0.000000e+00,True
5,tests_completed_count_bucket,2712.153341,2,0.000000e+00,True
7,first_completed_test_domain,2513.165260,8,0.000000e+00,True
6,first_test_domain,913.850909,8,5.808122e-192,True
4,tests_taken_count_bucket,865.278362,3,3.007531e-187,True
0,browser,295.630741,1,2.949230e-66,True
1,os_name_clean,256.330212,5,2.408203e-53,True
2,sub_region,155.040973,8,1.738967e-29,True


In [11]:
feature_to_inspect = "test_engagement_state"

ct = pd.crosstab(df[feature_to_inspect], df["has_signup"])
ct

has_signup,0,1
test_engagement_state,,
No Test,32608,48
Test Completed,416545,8640
Test Started Only,77511,39


In [12]:
(ct.div(ct.sum(axis=1), axis=0)
   .rename(columns={False: "no_signup_rate", True: "signup_rate"}))

has_signup,no_signup_rate,signup_rate
test_engagement_state,,
No Test,0.998530,0.001470
Test Completed,0.979679,0.020321
Test Started Only,0.999497,0.000503


### Interpretation Notes

- Chi-square tests evaluate whether signup behavior is independent of each feature.
- These tests are univariate and do not control for confounding variables.
- Statistical significance does not imply causality.
- Features identified as significant here will be further evaluated in:
  - Encoded Chi-square tests
  - Multivariate logistic regression